## 31. Design a **production-grade RAG system**

> **Answer:** “I would design RAG as an end-to-end pipeline covering **ingestion, processing, indexing, retrieval, reranking, generation, security, deployment, scalability, and observability**. For Azure, I would typically use Blob Storage, Document Intelligence where required, Azure AI Search, Azure OpenAI, and FastAPI.”

### Architecture

```text
Documents
   ↓
Blob Storage
   ↓
Document Processing / Chunking
   ↓
Embeddings
   ↓
Azure AI Search
   ↓
────────────────────────
      Query
        ↓
   Hybrid Search
        ↓
 Metadata Filter
        ↓
      Top-K
        ↓
     Reranking
        ↓
    Context
        ↓
   Azure OpenAI
        ↓
 Guardrails / Validation
        ↓
     Response
```

### Production design — 5 pillars

**Development**
- Chunking strategy
- Embeddings
- Hybrid search
- Metadata filtering
- Top-K + reranking
- Pydantic/structured output
- RAGAS + DeepEval

**Security**
- Entra ID / RBAC
- Managed Identity
- Key Vault
- Tenant-level filtering
- PII protection
- Prompt-injection protection
- Content Safety

**Deployment**
- FastAPI
- Docker
- API Management
- CI/CD
- Azure Container Apps / AKS

**Scalability**
- Async ingestion
- Queue-based processing
- Horizontal scaling
- Redis caching
- Batch embedding
- Incremental indexing
- Rate limiting

**Observability**
- LangSmith
- Application Insights
- Retrieval latency
- LLM latency
- Token usage
- Retrieval quality
- Failed queries
- RAGAS/DeepEval metrics

### Key interview point

> **“Production RAG is not just Vector DB + LLM. I need to optimize the complete lifecycle: document ingestion, chunking, embedding, hybrid retrieval, filtering, reranking, grounded generation, security, scalability, evaluation, and observability.”**

**Interview one-liner:**

> **“My production RAG architecture would use Blob Storage for ingestion, Azure AI Search for hybrid retrieval and reranking, Azure OpenAI for generation, and FastAPI as the application layer, with Entra ID, Key Vault, guardrails, async processing, monitoring, and RAG evaluation around the core pipeline.”**

## 32. How would you design RAG for **10 million documents**?

> **Answer:** “For 10 million documents, I would focus primarily on **distributed ingestion, scalable indexing, metadata partitioning, efficient retrieval, caching, and asynchronous processing**. I would not process all documents synchronously.”

### Architecture

```text
10M Documents
      ↓
Blob Storage
      ↓
Queue / Event Bus
      ↓
Async Processing Workers
      ↓
Chunking + Embeddings
      ↓
Distributed Search Index
      ↓
Query
      ↓
Metadata Filter
      ↓
Hybrid Search
      ↓
Top-K
      ↓
Reranking
      ↓
LLM
```

### Production approach

**Development**
- Batch/incremental ingestion
- Appropriate chunking
- Hybrid search
- Metadata filtering
- Top-K + reranking
- Document versioning

**Security**
- Tenant/document-level authorization
- Entra ID / RBAC
- Metadata-based access filtering
- PII protection

**Deployment**
- Containerized ingestion workers
- API Management + FastAPI
- Separate ingestion and query services

**Scalability**
- Async queue-based ingestion
- Parallel embedding generation
- Distributed search/indexing
- Horizontal worker scaling
- Redis/semantic caching
- Incremental updates instead of rebuilding the entire index
- Rate limiting

**Observability**
- Indexing success/failure
- Search latency
- Embedding latency
- Query latency
- Top-K retrieval quality
- Token usage and cost

### Important interview point

> **“At 10 million documents, the key is separating the ingestion and query paths. Ingestion should be asynchronous and horizontally scalable, while the query path should use metadata filtering and hybrid retrieval to reduce the search space before reranking and LLM generation.”**

**Interview one-liner:**

> **“For 10 million documents, I would use asynchronous distributed ingestion, incremental indexing, metadata partitioning, scalable hybrid search, reranking, caching, and horizontal scaling, rather than treating the RAG system as a single synchronous pipeline.”**

## 33. How do you choose a **chunking strategy**?

> **Answer:** “I choose chunking based on the **document structure, query pattern, and relationship between information**. My goal is to create chunks that are semantically meaningful, retain enough context, and remain small enough for efficient retrieval.”

### Decision approach

```text
Document Type
     ↓
Understand Structure
     ↓
Choose Chunking
```

| Document | Preferred approach |
|---|---|
| Normal text | Recursive chunking |
| Structured sections | Structure/heading-based |
| Large documents | Parent-child |
| Highly semantic content | Semantic chunking |
| Tables | Table-aware chunking |
| JSON / master data | Custom entity-based chunking |
| PDFs with images | Multimodal/document-aware processing |

### Important parameters

- **Chunk size** → based on content and model context
- **Chunk overlap** → preserve boundary context
- **Metadata** → document ID, page, section, tenant, etc.
- **Parent-child relationship** → retrieve small child chunks but provide larger parent context
- **Semantic boundaries** → avoid splitting related information

### Example

```text
200-page PDF

Parent Document
      ↓
Sections / Pages
      ↓
Child Chunks
      ↓
Embeddings
      ↓
Vector Search
      ↓
Retrieve Child
      ↓
Return Parent Context
      ↓
LLM
```

### Interview one-liner

> **“I don't use one chunking strategy for every document. I select it based on document structure and query behavior, and for enterprise RAG I often combine structure-aware or parent-child chunking with metadata so retrieval remains precise while preserving sufficient context.”**

## 34. Recursive vs Semantic vs Parent-Child Chunking?

> **Answer:** “The main difference is how they decide chunk boundaries and how much context they preserve. **Recursive chunking** is simple and structure-aware, **semantic chunking** groups text based on meaning, and **parent-child chunking** separates retrieval granularity from the context sent to the LLM.”

| Strategy | How it works | Best use case | Trade-off |
|---|---|---|---|
| **Recursive** | Splits using separators such as paragraphs, sentences, then words | General documents | Simple but may split semantic concepts |
| **Semantic** | Splits when semantic meaning changes | Complex narrative/knowledge documents | More computationally expensive |
| **Parent-Child** | Small child chunks for retrieval + larger parent context for generation | Large PDFs, manuals, policies | More complex indexing |

### Example

```text id="g6r8kn"
Recursive:
Document
 ↓
Paragraphs
 ↓
Chunks
 ↓
Embedding
```

```text id="wq7p6u"
Semantic:
Document
 ↓
Meaning-based sections
 ↓
Semantic Chunks
 ↓
Embedding
```

```text id="f6k0y1"
Parent-Child:

Parent Section
 ├── Child Chunk 1
 ├── Child Chunk 2
 └── Child Chunk 3

Search → Child 2
             ↓
       Return Parent
             ↓
            LLM
```

### Interview preference

> **“For a standard RAG application, I would start with recursive chunking. If semantic boundaries are important, I would use semantic chunking. For large enterprise documents where a small relevant section needs broader context, I prefer parent-child chunking.”**

**One-liner:**

> **“Recursive optimizes simplicity, semantic optimizes meaning, and parent-child optimizes retrieval precision versus contextual completeness.”**

## 35. How do you handle **tables and images in PDFs**?

> **Answer:** “I don't treat a PDF as plain text only. I use a document-intelligence pipeline to separately extract **text, tables, and images**. Text is chunked normally, tables are converted into structured representations, and images are processed using OCR or a vision model. I then store the extracted content with metadata such as page number and document ID for RAG retrieval.”

### Production flow

```text
PDF
 ↓
Document Intelligence
 ├── Text → Text Chunks → Embeddings
 ├── Tables → Structured/Markdown → Embeddings
 └── Images → OCR/Vision → Description → Embeddings
                         ↓
                  Azure AI Search
                         ↓
                    Hybrid Search
                         ↓
                       LLM
```

### Tables

```text
PDF Table
   ↓
Extract rows/columns
   ↓
Convert to Markdown/JSON
   ↓
Add metadata
   ↓
Embed + Index
```

For example:

```text
| Employee | Skill | Experience |
|----------|-------|------------|
| E101     | Java  | 5 years    |
```

I preserve the **column headers and row relationships** so retrieval doesn't lose the table's meaning.

### Images

```text
PDF Image
   ↓
OCR / Vision Model
   ↓
Text + Description
   ↓
Metadata
   ↓
Embedding
```

For charts/diagrams, I would use a **vision-capable model** rather than relying only on OCR.

### Important interview point

> **“For complex PDFs, I create separate processing paths for text, tables, and images, but index them into a common searchable knowledge layer with page-level metadata. This prevents information loss that would occur if I simply extracted the PDF as plain text.”**

**Interview one-liner:**

> **“Text goes through normal chunking, tables are converted to structured content, and images are processed through OCR or vision models; all three are indexed with metadata so the RAG system can retrieve the correct content and source page.”**

## 36. How do you handle a **200-page PDF**?

> **Answer:** “I would not send the entire 200-page PDF directly to the LLM. I would process it through a document-ingestion pipeline, extract text, tables and images, apply appropriate chunking, generate embeddings, and store the chunks with metadata such as document ID, page number and section. At query time, I would use hybrid search, metadata filtering, Top-K retrieval and reranking to retrieve only the relevant content.”

### Flow

```text
200-Page PDF
      ↓
Document Processing
      ↓
Text / Tables / Images
      ↓
Structure-Aware Chunking
      ↓
Embeddings
      ↓
Azure AI Search
      ↓
User Query
      ↓
Hybrid Search + Metadata Filter
      ↓
Top-K
      ↓
Reranking
      ↓
Relevant Context
      ↓
Azure OpenAI
      ↓
Answer + Citations
```

### Important point

If the question is:

> **“Information on page 2 is related to information on page 30. How do you handle it?”**

I would use **parent-child or hierarchical retrieval**, rather than treating every page independently.

```text
Document
   ↓
Parent Sections
   ↓
Child Chunks
   ↓
Retrieve relevant child chunks
   ↓
Expand to parent context
   ↓
LLM
```

**Interview one-liner:**

> **“For a 200-page PDF, I use hierarchical/parent-child chunking with page and section metadata, then retrieve only the relevant chunks using hybrid search and reranking instead of passing the entire document to the LLM.”**

## 37. How do you preserve relationships between chunks?

> **Answer:** “I preserve relationships using **hierarchical metadata and parent-child relationships** rather than treating every chunk independently. Each child chunk contains metadata such as `document_id`, `section_id`, `parent_id`, `page_number`, and `chunk_id`. During retrieval, I can retrieve a relevant child chunk and then expand to its parent or related chunks to provide sufficient context to the LLM.”

### Example

```text
Document
   │
   ├── Section 1
   │    ├── Chunk 1
   │    ├── Chunk 2
   │    └── Chunk 3
   │
   └── Section 2
        ├── Chunk 4
        └── Chunk 5
```

Metadata:

```python
{
    "document_id": "DOC01",
    "parent_id": "SECTION01",
    "chunk_id": "CHUNK02",
    "page": 30
}
```

### Retrieval

```text
Query
 ↓
Retrieve Chunk 2
 ↓
parent_id = SECTION01
 ↓
Retrieve parent / related chunks
 ↓
Combine context
 ↓
LLM
```

### Techniques

- **Parent-child relationships**
- **Document / section / page IDs**
- **Chunk sequence numbers**
- **Metadata filtering**
- **Hierarchical retrieval**
- **Neighbor-chunk expansion**
- **Semantic relationships**

**Interview one-liner:**

> **“I preserve chunk relationships through hierarchical metadata, parent-child IDs and chunk sequencing. At retrieval time, I can expand a relevant child chunk to its parent or neighboring chunks so the LLM receives the required context instead of an isolated fragment.”**

## 38. How do you choose **embedding models**?

> **Answer:** “I choose an embedding model based on **semantic retrieval quality, domain, language support, embedding dimension, latency, cost, and deployment constraints**. I benchmark candidate models on my own golden dataset using retrieval metrics such as Recall@K, Precision@K and MRR rather than choosing only based on model popularity.”

### Key factors

- **Domain** → general vs medical/legal/technical
- **Language** → English vs multilingual
- **Retrieval quality** → Recall@K, MRR, NDCG
- **Dimension** → storage and search performance
- **Latency** → embedding generation speed
- **Cost** → API vs self-hosted
- **Security** → cloud API vs local deployment
- **Consistency** → same embedding model for indexing and querying

**Interview one-liner:**

> **“I select embeddings empirically using my domain dataset, balancing retrieval quality, dimensions, latency, cost and language/domain requirements.”**

---

## 39. How do you decide **embedding dimensions**?

> **Answer:** “I don't choose the dimension independently. The dimension is determined by the embedding model or its configurable output size. If the model supports multiple dimensions, I benchmark the available sizes against retrieval quality, storage, latency and index performance.”

### Trade-off

```text
Higher Dimension
      ↓
More semantic representation
      ↓
Potentially better retrieval
      +
More storage / computation
```

```text
Lower Dimension
      ↓
Smaller vectors
      ↓
Lower storage / faster search
      +
Potential retrieval-quality loss
```

### Example

```text
1536 dimensions
→ 1536 floating-point values/vector

768 dimensions
→ 768 floating-point values/vector
```

For **10M documents**, this matters significantly because vector storage grows with:

```text
Number of vectors × Dimensions × Bytes per value
```

### Important interview point

> **“I would benchmark different supported dimensions using the same golden dataset and select the smallest dimension that meets my required retrieval-quality threshold while optimizing storage, latency and cost.”**

**Interview one-liner:**

> **“Embedding dimension is a quality-versus-cost trade-off; I choose it based on model capability and benchmark retrieval quality against storage, latency and scale requirements.”**

## 40. Vector Search vs Keyword Search vs Hybrid Search?

> **Answer:** “Keyword search matches exact terms, vector search matches semantic meaning, and hybrid search combines both to improve retrieval accuracy.”

| Search | How it works | Best for |
|---|---|---|
| **Keyword** | Exact/token-based matching, e.g. BM25 | IDs, names, codes, exact terms |
| **Vector** | Embedding similarity | Semantic meaning, paraphrases |
| **Hybrid** | Keyword + vector together | Enterprise RAG |

### Example

Query:

> **“Java engineer with 5 years experience”**

**Keyword search** can strongly match:

```text
"Java Engineer"
"5 years"
```

**Vector search** can understand:

```text
"Senior Java Developer with 5 years experience"
```

**Hybrid search** combines both signals.

---

## 41. Why use **Hybrid Search**?

> **Answer:** “I use hybrid search because enterprise queries contain both **semantic intent and exact business terms**. Vector search handles meaning, while keyword search handles exact entities such as employee IDs, product codes, policy numbers, names, and technical terms. Combining them generally gives more robust retrieval.”

### Flow

```text
User Query
    ↓
 ┌───────────────┐
 ↓               ↓
Keyword       Vector
Search        Search
 ↓               ↓
 └───────┬───────┘
         ↓
   Score Fusion
         ↓
      Top-K
         ↓
     Reranker
         ↓
       LLM
```

### Interview example

For:

> **“What is policy POL-12345 regarding diabetes treatment?”**

```text
Keyword → POL-12345        ← exact match
Vector  → diabetes treatment ← semantic match
              ↓
          Hybrid Search
```

**Interview one-liner:**

> **“I prefer hybrid search in production RAG because keyword search provides precision for exact terms while vector search provides semantic recall; combining them gives better retrieval robustness across both structured and natural-language queries.”**

## 42. How does **reranking** work?

> **Answer:** “Reranking is a second-stage retrieval process. The initial retriever gets a relatively large candidate set, such as Top-20 or Top-50, and a reranker evaluates the relevance of each candidate against the query and produces a better-ranked list. We then pass only the best results to the LLM.”

### Flow

```text
Query
  ↓
Hybrid / Vector Search
  ↓
Top-20 / Top-50 candidates
  ↓
Reranker
  ↓
Top-5 / Top-10
  ↓
LLM
```

### Why?

Initial retrieval is optimized for **speed and recall**.

Reranking is optimized for **precision**.

Example:

```text
Retriever → 20 candidates
                ↓
          Reranker scores
                ↓
        Best 5 documents
                ↓
              LLM
```

Common approaches include **cross-encoder rerankers** and managed search reranking capabilities.

---

## 43. Where should **reranking** happen in the RAG pipeline?

> **Answer:** “Reranking should happen **after the initial retrieval and before context is sent to the LLM**. I first retrieve a broader candidate set using vector or hybrid search, then rerank those candidates, and finally send the top-ranked chunks to the LLM.”

```text
User Query
    ↓
Query Processing
    ↓
Hybrid Search
    ↓
Metadata Filtering
    ↓
Candidate Top-K
    ↓
Reranking
    ↓
Final Top-K
    ↓
Context Assembly
    ↓
LLM
    ↓
Answer
```

### Interview one-liner

> **“Retrieval gives me high recall; reranking improves precision. Therefore, I place reranking between initial retrieval and LLM context construction.”**

## 44. How do you select **Top-K**?

> **Answer:** “I don't choose Top-K arbitrarily. I determine it through **retrieval evaluation and context-window constraints**. I start with a larger candidate K for recall, apply reranking, and then pass a smaller final K to the LLM based on relevance and available context.”

### Typical flow

```text
Query
 ↓
Hybrid Search → Top-20 / Top-50
 ↓
Reranking
 ↓
Final Top-5 / Top-10
 ↓
LLM
```

### Factors I consider

- **Recall requirement** → higher K improves the chance of finding relevant information.
- **Precision** → too many irrelevant chunks can degrade the answer.
- **Chunk size** → larger chunks require smaller K.
- **Context window** → total retrieved tokens must fit comfortably.
- **Reranker quality** → good reranking allows a smaller final K.
- **Latency & cost** → higher K increases search/reranking/LLM processing.
- **Evaluation dataset** → validate using Recall@K, Precision@K, MRR, NDCG and answer quality.

### Example

```text
Initial retrieval: K = 20
        ↓
Reranker
        ↓
Final context: K = 5
        ↓
LLM
```

If Recall@5 is poor but Recall@10 is significantly better, I may increase the final K to 10.

**Interview one-liner:**

> **“I select K empirically: retrieve enough candidates for high recall, rerank them, then choose the smallest final K that provides the required answer quality without unnecessarily increasing context, latency, and cost.”**

## 45. How do you handle **metadata filtering**?

> **Answer:** “I use metadata filtering to restrict retrieval to documents the user is authorized to access and to improve retrieval precision. I store metadata such as **tenant ID, document type, department, date, region, source, and access level** along with each chunk. At query time, I apply the relevant filters before or during vector/hybrid search.”

### Example

```text
User Query
   ↓
User Context
   ↓
Metadata Filter
   ↓
Hybrid / Vector Search
   ↓
Top-K
   ↓
Reranking
   ↓
LLM
```

Example metadata:

```python
{
    "tenant_id": "T001",
    "department": "HR",
    "document_type": "policy",
    "region": "India",
    "access_level": "manager"
}
```

Query:

```text
"Leave policy"
```

Filter:

```text
tenant_id = T001
AND department = HR
AND access_level = manager
```

### Why it is important

- **Security** → prevents cross-tenant/data leakage.
- **Precision** → searches only the relevant document subset.
- **Performance** → reduces the search space.
- **Compliance** → enforces document-level access policies.

**Interview one-liner:**

> **“I treat metadata filtering as both a retrieval optimization and a security control. I apply tenant, authorization, document-type and other business filters before retrieval so the vector search never considers documents the user shouldn't access.”**

## 46. How do you prevent irrelevant documents from reaching the LLM?

> **Answer:** “I use a multi-stage retrieval pipeline: **metadata filtering → hybrid search → Top-K retrieval → reranking → relevance threshold**. Only the highest-quality chunks that pass the relevance threshold are added to the LLM context.”

```text
Query
 ↓
Metadata Filter
 ↓
Hybrid Search
 ↓
Top-20
 ↓
Reranking
 ↓
Relevance Threshold
 ↓
Top-5
 ↓
LLM
```

**Interview one-liner:**

> **“I control context quality before the LLM using metadata filtering, hybrid retrieval, reranking and relevance thresholds, so irrelevant chunks are removed before context construction.”**

---

## 47. How do you handle **duplicate chunks**?

> **Answer:** “I detect duplicates during ingestion using a **content hash** or normalized-text hash. Before indexing, I check whether the same content already exists and avoid creating duplicate vectors. I can also apply deduplication during retrieval using document ID or content similarity.”

```text
Document
 ↓
Normalize Text
 ↓
Generate Hash
 ↓
Already Exists?
 ├── Yes → Skip
 └── No  → Embed → Index
```

Example:

```python
import hashlib

chunk_hash = hashlib.sha256(
    chunk_text.strip().lower().encode()
).hexdigest()
```

**Interview one-liner:**

> **“I primarily handle duplicates during ingestion using content hashing, and I can additionally deduplicate retrieved results before sending context to the LLM.”**

---

## 48. How do you handle **stale documents**?

> **Answer:** “I maintain document **versioning and metadata such as `updated_at`, version, status and document ID**. When a document changes, I reprocess and re-index the affected chunks and mark the previous version inactive or delete it. At retrieval time, I filter out inactive or expired versions.”

```text
Old Document
    ↓
Document Updated
    ↓
Detect Change
    ↓
Reprocess
    ↓
New Version → Index
    ↓
Old Version → Inactive/Delete
```

### Production approach

- Document versioning
- `updated_at` metadata
- Active/inactive status
- Incremental re-indexing
- Delete obsolete vectors
- Event-driven update pipeline
- Retrieval-time filtering

**Interview one-liner:**

> **“I use versioned documents and incremental re-indexing. When a source changes, I update only the affected chunks, deactivate the old version, and ensure retrieval filters out stale content.”**

## 49. How do you implement **document versioning**?

> **Answer:** “I maintain a version number and metadata for every document. When the source document changes, I create a new version, reprocess its chunks, and mark the previous version as inactive. Retrieval only considers the active version.”

```text
Document v1
   ↓
Updated
   ↓
Document v2
   ↓
New Chunks → Re-embed → Index
   ↓
v1 → Inactive
```

Metadata:

```python
{
    "document_id": "DOC101",
    "version": 2,
    "status": "active",
    "updated_at": "2026-08-16"
}
```

**Interview one-liner:**

> **“I use document ID + version + status metadata, create a new version when content changes, and ensure retrieval only uses the active version.”**

---

## 50. How do you implement **incremental document ingestion**?

> **Answer:** “I avoid reprocessing the entire knowledge base. I detect newly added or modified documents using **document IDs, timestamps, versions, or content hashes**, and process only those documents. I typically trigger the pipeline asynchronously through an event or queue.”

```text
New / Updated Document
        ↓
Change Detection
        ↓
Queue / Event
        ↓
Chunk
        ↓
Embed
        ↓
Index / Update
```

### Example

```text
10M documents
      ↓
100 documents changed
      ↓
Process only 100
```

**Interview one-liner:**

> **“I implement incremental ingestion using change detection and event-driven processing, so only new or modified documents are chunked, embedded and indexed instead of rebuilding the entire index.”**

---

## 51. How do you handle **deleted documents**?

> **Answer:** “I propagate the deletion from the source system to the search index. Using the document ID, I identify all associated chunks and either delete them or mark them inactive. I also make sure deleted documents cannot be returned by retrieval.”

```text
Source Document Deleted
        ↓
Delete Event
        ↓
Find document_id
        ↓
Delete / Deactivate
all associated chunks
        ↓
Search Index Updated
```

### Important

Every chunk should carry:

```python
{
    "document_id": "DOC101",
    "chunk_id": "DOC101-C05"
}
```

So deletion becomes:

```text
document_id = DOC101
        ↓
Delete DOC101-C01
Delete DOC101-C02
Delete DOC101-C03
...
```

**Interview one-liner:**

> **“I propagate deletion events using the document ID, remove or deactivate all associated chunks from the vector/search index, and verify that inactive documents are excluded from retrieval.”**

## 52. How do you implement **citations in RAG**?

> **Answer:** “I preserve source metadata during ingestion, such as **document ID, page number, section, and source URL**. When chunks are retrieved, I pass this metadata along with the context to the LLM and return the corresponding sources with the generated answer.”

```text
Document
 ↓
Chunk + Metadata
 ↓
Vector / Search Index
 ↓
Retrieve
 ↓
Chunk + Source Metadata
 ↓
LLM
 ↓
Answer + Citations
```

Example:

```python
{
    "text": "...",
    "document_id": "policy_101",
    "page": 25,
    "source": "HR_Policy.pdf"
}
```

**Interview one-liner:**

> **“I implement citations by preserving source metadata with every chunk and returning the metadata of the chunks actually used to generate the answer.”**

---

## 53. How do you prevent **hallucinations in RAG**?

> **Answer:** “I ground the LLM strictly on retrieved context and use multiple controls: **high-quality retrieval, reranking, relevance thresholds, prompt grounding instructions, structured output validation, and evaluation for faithfulness**. If sufficient evidence is not retrieved, I don't allow the model to confidently answer from its own knowledge.”

```text
Query
 ↓
Hybrid Retrieval
 ↓
Reranking
 ↓
Relevance Threshold
 ↓
Sufficient Context?
 ├── Yes → LLM → Answer
 └── No  → "I don't know"
```

### Key controls

- Good chunking
- Hybrid search
- Metadata filtering
- Reranking
- Relevance threshold
- Grounded prompt
- Citation verification
- RAGAS/DeepEval
- "I don't know" fallback

**Interview one-liner:**

> **“I reduce hallucination by ensuring the answer is grounded in high-quality retrieved context and by refusing to answer when sufficient evidence is not available.”**

---

## 54. How do you handle **"I don't know" responses**?

> **Answer:** “I define an explicit **confidence/relevance threshold** before generation. If retrieval doesn't return sufficiently relevant context, I don't send weak context to the LLM. Instead, I return a controlled response such as *‘I don't have sufficient information to answer this question.’*”

### Flow

```text
Query
 ↓
Retrieve Top-K
 ↓
Rerank
 ↓
Relevance Score
 ↓
Score >= Threshold?
 ├── Yes → LLM → Answer + Citation
 └── No  → I Don't Know
```

### Example

```python
if best_score < MIN_RELEVANCE:
    return "I don't have sufficient information to answer this."

return generate_answer(context)
```

**Interview one-liner:**

> **“I treat ‘I don't know’ as a valid outcome. If retrieval confidence is below the configured threshold or the context doesn't support the question, I return a controlled fallback instead of allowing the LLM to guess.”**